<a href="https://colab.research.google.com/github/susmitsingh01/llm-inference-cookbook/blob/main/Inference_Cookbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import sys
import torch
import subprocess

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

print("\n--- nvidia-smi ---")
subprocess.run("nvidia-smi", shell=True)

print("\n--- nvcc ---")
subprocess.run("nvcc --version", shell=True)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.9.0+cu126
Torch CUDA: 12.6
CUDA available: True
GPU: NVIDIA L4
Capability: (8, 9)

--- nvidia-smi ---

--- nvcc ---


CompletedProcess(args='nvcc --version', returncode=0)

In [10]:
!nvidia-smi

Mon Jun  1 02:59:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
# Core dependency repair
!pip install -q "requests==2.32.4"
!pip install -q "fsspec==2025.3.0"
!pip install -q "huggingface-hub>=0.34.0,<1.0"
!pip install -q "tokenizers>=0.22.0,<=0.23.0"
!pip install -q "safetensors>=0.4.3"

# Hugging Face stack
!pip install -q "transformers>=4.56.1,<=4.57.6"
!pip install -q "accelerate>=1.6.0,<=1.12.0"
!pip install -q datasets optimum

# Quantization package
!pip install -q bitsandbytes

# Flash Attention
!pip install -q flash-attn==2.8.3 --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 170.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [11]:
import importlib
import importlib.metadata

def check_package(display_name, import_name=None, pip_name=None):
    if import_name is None:
        import_name = display_name
    if pip_name is None:
        pip_name = display_name

    try:
        module = importlib.import_module(import_name)

        try:
            version = getattr(module, "__version__")
        except AttributeError:
            version = importlib.metadata.version(pip_name)

        print(f"✅ {display_name:<18} {version}")

    except Exception as e:
        print(f"❌ {display_name:<18} {type(e).__name__}: {e}")


packages = [
    ("transformers", "transformers", "transformers"),
    ("accelerate", "accelerate", "accelerate"),
    ("datasets", "datasets", "datasets"),
    ("optimum", "optimum", "optimum"),
    ("bitsandbytes", "bitsandbytes", "bitsandbytes"),
    ("flash-attn", "flash_attn", "flash-attn"),
    ("huggingface-hub", "huggingface_hub", "huggingface-hub"),
    ("tokenizers", "tokenizers", "tokenizers"),
    ("safetensors", "safetensors", "safetensors"),
    ("requests", "requests", "requests"),
    ("fsspec", "fsspec", "fsspec"),
]

for display_name, import_name, pip_name in packages:
    check_package(display_name, import_name, pip_name)

✅ transformers       4.57.6
✅ accelerate         1.12.0
✅ datasets           4.0.0
❌ optimum            ModuleNotFoundError: No module named 'optimum'
❌ bitsandbytes       ModuleNotFoundError: No module named 'bitsandbytes'
❌ flash-attn         ModuleNotFoundError: No module named 'flash_attn'
✅ huggingface-hub    0.36.0
✅ tokenizers         0.22.2
✅ safetensors        0.7.0
✅ requests           2.32.4
✅ fsspec             2025.3.0


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Create Project Folder Structure on Drive

In [ ]:
import os

# ── Base folder on Drive ───────────────────────────────────────────────────────
DRIVE_BASE    = "/content/drive/MyDrive/llm-inference-cookbook"
RESULTS_DIR   = f"{DRIVE_BASE}/results"
CHARTS_DIR    = f"{DRIVE_BASE}/results/charts"
NOTEBOOKS_DIR = f"{DRIVE_BASE}/notebooks"

for folder in [DRIVE_BASE, RESULTS_DIR, CHARTS_DIR, NOTEBOOKS_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ {folder}")

✅ /content/drive/MyDrive/llm-inference-cookbook
✅ /content/drive/MyDrive/llm-inference-cookbook/results
✅ /content/drive/MyDrive/llm-inference-cookbook/results/charts
✅ /content/drive/MyDrive/llm-inference-cookbook/notebooks


# Imports and Constants

In [ ]:
import torch
import time
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import pynvml

# ── Model identifiers ──────────────────────────────────────────────────────────
MODEL_ID       = "meta-llama/Meta-Llama-3-8B-Instruct"
DRAFT_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# ── Evaluation dataset ─────────────────────────────────────────────────────────
EVAL_DATASET = "wikitext"
EVAL_CONFIG  = "wikitext-2-raw-v1"
EVAL_SAMPLES = 100

# ── Standard prompts used across ALL experiments ───────────────────────────────
# Using the same prompts everywhere makes numbers directly comparable
BENCHMARK_PROMPTS = {
    "short":   "What is the capital of France?",
    "medium":  "Explain the water cycle in detail.",
    "long":    "Write a comprehensive overview of the history of artificial intelligence.",
    "code":    "Write a Python function that implements binary search with comments.",
    "factual": "What are the main differences between supervised and unsupervised learning?"
}

print("✅ Imports and constants ready")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"VRAM total     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch        : {torch.__version__}")

✅ Imports and constants ready
CUDA available : True
GPU            : NVIDIA L4
VRAM total     : 23.7 GB
PyTorch        : 2.9.0+cu126


# GPU Memory Utility

In [ ]:
# Initialize NVML — NVIDIA's management library for querying GPU stats
pynvml.nvmlInit()
_nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_gpu_memory_gb():
    """Returns currently used GPU memory in GB."""
    info = pynvml.nvmlDeviceGetMemoryInfo(_nvml_handle)
    return round(info.used / 1e9, 2)

# Measurement Functions

In [ ]:
def measure_throughput(model, tokenizer, prompt, max_new_tokens=200, batch_size=1, n_runs=3):
    """
    Measures tokens generated per second.
    Runs n_runs times and returns mean ± std to show stability.
    torch.cuda.synchronize() is critical — without it, timing is wrong
    because GPU ops are async and the timer would stop before GPU finishes.
    """
    inputs = tokenizer(
        [prompt] * batch_size,
        return_tensors="pt",
        padding=True
    ).to("cuda")

    timings = []
    with torch.no_grad():
        for _ in range(n_runs):
            torch.cuda.synchronize()        # wait for GPU to finish any pending work
            t0 = time.perf_counter()
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            torch.cuda.synchronize()        # wait for generation to fully complete
            t1 = time.perf_counter()

            # Only count NEW tokens, not the prompt tokens
            new_tokens = (out.shape[1] - inputs["input_ids"].shape[1]) * batch_size
            timings.append(new_tokens / (t1 - t0))

    return {
        "mean_tok_per_sec": round(float(np.mean(timings)), 2),
        "std_tok_per_sec":  round(float(np.std(timings)), 2)
    }


def measure_ttft(model, tokenizer, prompt):
    """
    Measures Time To First Token (TTFT) in milliseconds.
    TTFT = how long the user waits before seeing ANY output.
    This is dominated by the prefill pass (processing the entire prompt at once).
    We generate exactly 1 token to isolate prefill time from decode time.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    times = []
    with torch.no_grad():
        for _ in range(3):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model.generate(**inputs, max_new_tokens=1, do_sample=False)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append((t1 - t0) * 1000)

    return round(float(np.mean(times)), 2)  # milliseconds


def measure_tpot(model, tokenizer, prompt, max_new_tokens=100):
    """
    Measures Time Per Output Token (TPOT) in milliseconds.
    TPOT = average time to generate each token after the first.
    This is the decode phase — one forward pass per token.
    Lower TPOT = faster streaming experience for the user.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
        t1 = time.perf_counter()

    n_new = out.shape[1] - inputs["input_ids"].shape[1]
    return round((t1 - t0) * 1000 / max(n_new, 1), 2)  # ms per token


def measure_perplexity(model, tokenizer, n_samples=EVAL_SAMPLES):
    """
    Measures perplexity on WikiText-2 test set.
    Perplexity = how 'surprised' the model is by real text.
    Lower = better. FP16 baseline should be ~8.5.
    We use this to check that quantization hasn't hurt quality.
    Formula: exp(mean(negative log likelihoods))
    We truncate to 512 tokens per sample to keep runtime reasonable.
    """
    dataset = load_dataset(EVAL_DATASET, EVAL_CONFIG, split="test")
    texts = [x for x in dataset["text"] if len(x.strip()) > 50][:n_samples]

    nlls = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=512
            ).to("cuda")

            if inputs["input_ids"].shape[1] < 2:
                continue

            out = model(**inputs, labels=inputs["input_ids"])
            nlls.append(out.loss.item())

    return round(float(np.exp(np.mean(nlls))), 4)

# Save Utility

In [ ]:
def save_result(result_dict):
    """
    Appends one experiment's results to all_numbers.json on Google Drive.
    We use append-not-overwrite so every experiment accumulates in one file.
    The README table and waterfall chart will be generated FROM this file,
    so keeping it clean and complete is important.
    """
    path = os.path.join(RESULTS_DIR, "all_numbers.json")

    # Load existing data if file exists, otherwise start fresh
    if os.path.exists(path):
        with open(path, "r") as f:
            data = json.load(f)
    else:
        data = []

    # Stamp every result with a timestamp for debugging
    result_dict["timestamp"] = time.strftime("%Y-%m-%dT%H:%M:%S")
    data.append(result_dict)

    with open(path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"✅ Saved to Drive: {path}")
    print(f"   Total results in file: {len(data)}")

# HuggingFace Login & Load Model

In [ ]:
from huggingface_hub import login

# You need:
# 1. A HuggingFace account
# 2. License accepted at huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
# 3. An access token from huggingface.co/settings/tokens
login()  # will prompt for token

In [ ]:
# Clear any leftover GPU memory from previous runs
torch.cuda.empty_cache()

print(f"VRAM before load : {get_gpu_memory_gb():.2f} GB")
print("Loading Llama 3 8B FP16 — expect ~2 minutes...\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token  # required for batching in Exp 05

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # FP16 = 2 bytes per param → ~16GB for 8B model
    device_map="cuda"            # puts entire model on GPU
)
model.eval()  # disables dropout — we're doing inference not training

print(f"\n✅ Model loaded")
print(f"   Parameters : {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"   VRAM used  : {get_gpu_memory_gb():.2f} GB  (expected ~16 GB)")

VRAM before load : 0.50 GB
Loading Llama 3 8B FP16 — expect ~2 minutes...



tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]